# Auction-consistent implied WACC sensitivity (reviewer-response only)

This notebook is a **standalone sensitivity / bounding exercise**.

- It **does not replace** the baseline methodology in `cost-of-capital.ipynb` and `lcoe.ipynb`.
- It keeps baseline non-financing inputs fixed and treats awarded auction prices as a benchmark target.
- The solved implied WACC is an **auction-consistent lower-bound/sensitivity case**, not a new true-WACC estimate.

## Scope and baseline logic reused

This notebook reproduces only the minimum logic needed from the baseline pipeline:

1. Baseline project-level inputs and outputs are read from `lcoe_solar_analysis.csv` (produced by `lcoe.ipynb`).
2. LCOE under financing is evaluated with the same discounted-terms structure used in `lcoe.ipynb`:
   - discounted energy denominator
   - discounted OPEX numerator component
   - CAPEX allocated over discounted energy
3. The observed awarded auction price (`sale_price_auction`) is used only as a target for a root-finding sensitivity.

In [ ]:
import ast
import csv
import math
from pathlib import Path

INPUT_CSV = Path('lcoe_solar_analysis.csv')
OUTPUT_CSV = Path('auction_consistent_wacc_results.csv')

LIFETIME_YEARS = 25
MIN_POSITIVE_WACC = 1e-6   # decimal
MAX_WACC = 1.00            # decimal (100%)
MAX_ITER = 200
TOL = 1e-8

In [ ]:
def parse_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return math.nan


def parse_opex_list(value):
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [float(x) for x in parsed]
    except (ValueError, SyntaxError, TypeError):
        pass
    return []


def modeled_lcoe(exp_energy_prod, capex, opex_inflated, wacc_decimal, lifetime=LIFETIME_YEARS):
    if not (exp_energy_prod > 0 and capex >= 0 and wacc_decimal > -1):
        return math.nan

    discounted_energy = 0.0
    discounted_opex = 0.0

    for t in range(1, lifetime + 1):
        denom = (1.0 + wacc_decimal) ** t
        discounted_energy += exp_energy_prod / denom
        opex_t = opex_inflated[t - 1] if t - 1 < len(opex_inflated) else 0.0
        discounted_opex += opex_t / denom

    if discounted_energy <= 0:
        return math.nan

    capex_component = capex / discounted_energy
    opex_component = discounted_opex / discounted_energy
    return capex_component + opex_component


def gap_function(wacc_decimal, exp_energy_prod, capex, opex_inflated, auction_price):
    return modeled_lcoe(exp_energy_prod, capex, opex_inflated, wacc_decimal) - auction_price

In [ ]:
def solve_implied_wacc(exp_energy_prod, capex, opex_inflated, auction_price,
                       low=MIN_POSITIVE_WACC, high=MAX_WACC,
                       max_iter=MAX_ITER, tol=TOL):
    # Guardrails for economically meaningful cases
    if not (auction_price > 0 and exp_energy_prod > 0 and capex >= 0 and len(opex_inflated) > 0):
        return math.nan, 'invalid_inputs'

    g_low = gap_function(low, exp_energy_prod, capex, opex_inflated, auction_price)
    g_high = gap_function(high, exp_energy_prod, capex, opex_inflated, auction_price)

    if math.isnan(g_low) or math.isnan(g_high):
        return math.nan, 'model_error'

    # Exact hits at bounds
    if abs(g_low) <= tol:
        return low * 100.0, 'solved_at_lower_bound'
    if abs(g_high) <= tol:
        return high * 100.0, 'solved_at_upper_bound'

    # No positive root in bracket
    if g_low * g_high > 0:
        return math.nan, 'no_positive_solution_in_bounds'

    a, b = low, high
    fa, fb = g_low, g_high

    for _ in range(max_iter):
        m = 0.5 * (a + b)
        fm = gap_function(m, exp_energy_prod, capex, opex_inflated, auction_price)

        if math.isnan(fm):
            return math.nan, 'model_error'

        if abs(fm) <= tol or abs(b - a) <= tol:
            return m * 100.0, 'solved'

        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm

    return math.nan, 'not_converged'

In [ ]:
rows = []
with INPUT_CSV.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        exp_energy = parse_float(row.get('exp_energy_prod'))
        capex = parse_float(row.get('capex'))
        opex_inflated = parse_opex_list(row.get('opex_inflated', '[]'))
        auction_price = parse_float(row.get('sale_price_auction'))

        baseline_wacc = parse_float(row.get('cost_of_capital'))
        baseline_lcoe = parse_float(row.get('lcoe_wacc'))
        baseline_lcoe_nonfin = parse_float(row.get('lcoe_baseline'))

        implied_wacc_pct, status = solve_implied_wacc(exp_energy, capex, opex_inflated, auction_price)

        implied_lcoe = math.nan
        implied_fin_component = math.nan
        implied_fin_share = math.nan

        if not math.isnan(implied_wacc_pct):
            implied_lcoe = modeled_lcoe(exp_energy, capex, opex_inflated, implied_wacc_pct / 100.0)
            implied_fin_component = implied_lcoe - baseline_lcoe_nonfin if not math.isnan(baseline_lcoe_nonfin) else math.nan
            implied_fin_share = (implied_fin_component / implied_lcoe) if (implied_lcoe and implied_lcoe > 0 and not math.isnan(implied_fin_component)) else math.nan

        baseline_fin_component = baseline_lcoe - baseline_lcoe_nonfin if not (math.isnan(baseline_lcoe) or math.isnan(baseline_lcoe_nonfin)) else math.nan
        baseline_fin_share = (baseline_fin_component / baseline_lcoe) if (baseline_lcoe and baseline_lcoe > 0 and not math.isnan(baseline_fin_component)) else math.nan

        rows.append({
            'power_plant_name': row.get('power_plant_name'),
            'date': row.get('date_x'),
            'year': row.get('year'),
            'month': row.get('month'),
            'baseline_estimated_wacc_pct': baseline_wacc,
            'implied_auction_consistent_wacc_pct': implied_wacc_pct,
            'baseline_modeled_lcoe_brl_mwh': baseline_lcoe,
            'observed_auction_price_brl_mwh': auction_price,
            'financing_component_baseline_brl_mwh': baseline_fin_component,
            'financing_share_baseline': baseline_fin_share,
            'financing_component_implied_brl_mwh': implied_fin_component,
            'financing_share_implied': implied_fin_share,
            'solution_status': status,
        })

len(rows), rows[0]

In [ ]:
fieldnames = [
    'power_plant_name', 'date', 'year', 'month',
    'baseline_estimated_wacc_pct',
    'implied_auction_consistent_wacc_pct',
    'baseline_modeled_lcoe_brl_mwh',
    'observed_auction_price_brl_mwh',
    'financing_component_baseline_brl_mwh',
    'financing_share_baseline',
    'financing_component_implied_brl_mwh',
    'financing_share_implied',
    'solution_status'
]

with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f'Saved: {OUTPUT_CSV} (rows={len(rows)})')

In [ ]:
# Quick status summary for reviewer response text
summary = {}
for r in rows:
    summary[r['solution_status']] = summary.get(r['solution_status'], 0) + 1

summary

## Interpretation note

This output should be interpreted as a **sensitivity bound**:

- It answers: "what WACC would make the model exactly match the awarded auction price, holding all non-financing assumptions fixed?"
- It does **not** claim that the solved implied WACC equals the project's true financing cost.
- Cases with `solution_status != solved` have no valid positive root within the configured economically meaningful bounds and are left as `NaN`.